# Stage 04-CF — Causal Forest Extension

Estimate causal effect and heterogeneous treatment effects using a Causal Forest (EconML).
Follow `skills/stage_04_cf.md` for detailed instructions.

This notebook is used by `/recast-cf` instead of `04_dml_extension.ipynb`.
It does NOT run DoubleML.

In [ ]:
from paths import *
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LassoCV

df     = pd.read_parquet(str(DATASET_PATH))
spec   = json.loads(PAPER_SPEC.read_text())
rep    = json.loads(REPLICATION_RESULTS.read_text())
config = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())

outcome    = spec['identification']['outcome_variable']
treatment  = spec['identification']['treatment_variable']
instrument = spec['identification'].get('instrument')
controls   = spec['identification']['controls']
id_type    = spec['identification']['type']

cf_config = config.get('causal_forest', {})
cf_method = cf_config.get('method', 'CausalForestDML')
if id_type == 'IV' and cf_method not in ('CausalIVForest', 'CausalForestDML'):
    cf_method = 'CausalIVForest'  # default for IV papers

print(f'Causal Forest method: {cf_method}  |  ID type: {id_type}')
print(f'Outcome: {outcome}  |  Treatment: {treatment}  |  Instrument: {instrument}')

In [ ]:
# --- AGENT: prepare data ---
# Drop rows with missing values in key variables
# Build arrays for EconML: X_cf (controls), Y_cf (outcome), T_cf (treatment), Z_cf (instrument)
# See skills/stage_04_cf.md section 2

key_cols = [outcome, treatment] + controls + ([instrument] if instrument else [])
df_clean = df[key_cols].dropna().reset_index(drop=True)
print(f'Clean dataset: {df_clean.shape}')

X_cf = df_clean[controls].values
Y_cf = df_clean[outcome].values
T_cf = df_clean[treatment].values
if instrument:
    Z_cf = df_clean[instrument].values
else:
    Z_cf = None

In [ ]:
# --- AGENT: fit causal forest ---
# Choose method based on cf_method (CausalForestDML, CausalIVForest, or CausalForest)
# Use hyperparameters from cf_config
# Extract: ATE with inference, per-observation CATEs with CIs, feature importances
# See skills/stage_04_cf.md sections 3a/3b/3c

# cf_ate = float(np.mean(cate_pred))
# cf_ate_ci_lo = float(np.mean(cate_ci_lo))  # from predict_interval()
# cf_ate_ci_hi = float(np.mean(cate_ci_hi))
# cf_ate_se = (cf_ate_ci_hi - cf_ate_ci_lo) / (2 * 1.96)  # back out SE from CI
# NOTE: Do NOT use np.std(cate_pred)/np.sqrt(n) -- that gives implausibly narrow SEs
# cate_pred = ...  (n_obs,)
# cate_ci_lo, cate_ci_hi = ...  (n_obs,) each
# feat_imp = ...  dict or array
pass

In [ ]:
# --- AGENT: write causal_forest_results.json ---
# See skills/stage_04_cf.md section 4 for schema

# Determine sign change vs published
published_coef = rep[0]['coef'] if isinstance(rep, list) else rep.get('coef', 0)
sign_change = bool(np.sign(cf_ate) != np.sign(published_coef))

# Normalize feature importances
# feat_imp_dict = {controls[i]: float(feat_imp[i]) for i in range(len(controls))}

cf_results = {
    'method': cf_method,
    'n_obs': int(df_clean.shape[0]),
    'n_estimators': cf_config.get('n_estimators', 1000),
    'honest': True,
    'ate': None,       # agent fills
    'ate_se': None,    # agent fills
    'ate_ci_lo': None, # agent fills
    'ate_ci_hi': None, # agent fills
    'cate_summary': {
        'mean': None,
        'sd': None,
        'min': None,
        'max': None,
        'pct_negative': None,
        'pct_significant': None
    },
    'feature_importances': {},  # agent fills
    'sign_change_vs_published': sign_change
}

CF_RESULTS = RESULTS_DIR / 'causal_forest_results.json'
CF_RESULTS.write_text(json.dumps(cf_results, indent=2))
print(f'✓ {CF_RESULTS}')

In [ ]:
# --- AGENT: group-level heterogeneity (GATE-equivalent) ---
# Group CATEs by quartiles of first control variable
# Write hte_results.json
# See skills/stage_04_cf.md section 5

HTE_RESULTS = RESULTS_DIR / 'hte_results.json'
# HTE_RESULTS.write_text(json.dumps(hte_results, indent=2))
pass

In [ ]:
# --- AGENT: forest plot ---
# Coefficient comparison: Published OLS/IV, Replicated, Causal Forest ATE
# Save to FIGURES_DIR / 'forest_plot.pdf' and 'forest_plot.png'
# See skills/stage_04_cf.md section 6
pass

In [ ]:
# --- AGENT: CATE histogram ---
# Distribution of individual-level CATE estimates
# Save to FIGURES_DIR / 'cate_histogram.pdf' and 'cate_histogram.png'
# See skills/stage_04_cf.md section 7
pass

In [ ]:
# --- AGENT: feature importance plot ---
# Horizontal bar chart of heterogeneity drivers
# Save to FIGURES_DIR / 'feature_importance.pdf' and 'feature_importance.png'
# See skills/stage_04_cf.md section 8
pass

In [ ]:
# --- AGENT: GATE plot ---
# Group-level treatment effects plot
# Save to FIGURES_DIR / 'gate_plot.pdf' and 'gate_plot.png'
# See skills/stage_04_cf.md section 9
pass

In [ ]:
# --- AGENT: LaTeX comparison table ---
# table_cf.tex: Published OLS | Published IV | Replicated | Causal Forest ATE
# Save to TABLES_DIR / 'table_cf.tex'
# See skills/stage_04_cf.md section 10
pass

In [ ]:
# --- AGENT: GATE LaTeX table ---
# table_gate.tex: group-level treatment effects
# Save to TABLES_DIR / 'table_gate.tex'
# See skills/stage_04_cf.md section 11
pass